In [1]:


import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report
import mlflow
import dagshub
import os
import joblib
import numpy as np


if not os.environ.get("DAGSHUB_USER") or not os.environ.get("DAGSHUB_TOKEN") or not os.environ.get("MLFLOW_TRACKING_URI"):
    print("Error: DagsHub credentials (DAGSHUB_USER, DAGSHUB_TOKEN) or MLFLOW_TRACKING_URI are not set.")
    print("Please set them as environment variables in your terminal before running this notebook.")
    
else:
    print(f"MLflow tracking URI set to: {os.environ['MLFLOW_TRACKING_URI']}")


try:
    dagshub.init(repo_owner='sandhya-bdb', repo_name='mlflow_dagshub_new', mlflow=True)
   
    mlflow.set_experiment("Beverage Price Range Prediction")
    print("DagsHub and MLflow initialized successfully.")
except Exception as e:
    print(f"Error initializing DagsHub/MLflow: {e}")
    

DATA_FILE = 'survey_results.csv'
TARGET_COLUMN = 'price_range'
RESPONDENT_ID_COLUMN = 'respondent_id'


BEST_MODEL_FILENAME = 'best_price_range_model.pkl'


try:
    df = pd.read_csv(DATA_FILE)
    print(f"Dataset '{DATA_FILE}' loaded successfully. Shape: {df.shape}")
    print("First 5 rows of the dataset:")
    print(df.head())
except FileNotFoundError:
    print(f"Error: The file '{DATA_FILE}' was not found. Please ensure it's in the same directory as the notebook or provide the full path.")
    exit() # Exit if data loading fails
except Exception as e:
    print(f"An error occurred during data loading: {e}")
    exit()


if TARGET_COLUMN not in df.columns:
    print(f"Error: Target column '{TARGET_COLUMN}' not found in the dataset.")
    exit()
X = df.drop(columns=[TARGET_COLUMN, RESPONDENT_ID_COLUMN], errors='ignore') # Ignore respondent_id if it exists
y = df[TARGET_COLUMN]

print(f"\nFeatures shape: {X.shape}, Target shape: {y.shape}")



X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.25, random_state=42)
print(f"Data split into training and testing sets.")
print(f"Training set features shape: {X_train.shape}")
print(f"Testing set features shape: {X_test.shape}")


models = {
    "Gaussian Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Support Vector Machine (SVM)": SVC(probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "Light Gradient Boosting Machine (LightGBM)": LGBMClassifier(random_state=42)
}

best_model = None
best_accuracy = -1
best_model_name = ""
best_run_id = "" # To store the MLflow run ID of the best model

print("\n--- Starting Model Training and Evaluation ---")


for model_name, model_instance in models.items():
    
    with mlflow.start_run(run_name=f"Train_{model_name}") as run: # Capture run object
        print(f"\nTraining {model_name}...")
        current_run_id = run.info.run_id # Get the ID of the current run
        print(f"MLflow Run ID: {current_run_id}")

       
        mlflow.log_params(model_instance.get_params())

      
        model_instance.fit(X_train, y_train)

     
        y_pred = model_instance.predict(X_test)

      
        accuracy = accuracy_score(y_test, y_pred)
        class_report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
        class_report_str = classification_report(y_test, y_pred, zero_division=0)

       
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("test_set_accuracy", accuracy)

        
        for class_label, metrics in class_report_dict.items():
            if isinstance(metrics, dict):
                mlflow.log_metric(f"precision_{class_label}", metrics.get("precision", 0))
                mlflow.log_metric(f"recall_{class_label}", metrics.get("recall", 0))
                mlflow.log_metric(f"f1-score_{class_label}", metrics.get("f1-score", 0))
            elif isinstance(metrics, (int, float)) and class_label != 'accuracy':
                 mlflow.log_metric(f"{class_label}_avg", metrics)

        
        report_filename = f"{model_name}_classification_report.txt"
        with open(report_filename, "w") as f:
            f.write(class_report_str)
        mlflow.log_artifact(report_filename)
        os.remove(report_filename) # Clean up local file

        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Classification Report:\n{class_report_str}")
        print("-" * 30)

       
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_model = model_instance
            best_model_name = model_name
            best_run_id = current_run_id # Store the MLflow run ID

print("\n--- Model Training and Evaluation Complete ---")
print(f"Best model found: '{best_model_name}' with accuracy {best_accuracy:.4f} (MLflow Run ID: {best_run_id})")


print("\n--- Saving Best Model and Preprocessing Artifacts ---")


# Define your filenames (these should be consistent with your Streamlit app)
BEST_MODEL_FILENAME = 'best_price_range_model.pkl'
# --- Saving Best Model ---
try:
    joblib.dump(best_model, BEST_MODEL_FILENAME)
    print(f"Best model saved to '{BEST_MODEL_FILENAME}'")
except Exception as e:
    print(f"Error saving best model: {e}")


# --- MLflow Logging (Ensure preprocessor is logged) ---
if best_run_id: # Assuming best_run_id is defined from your MLflow tracking
    try:
        print(f"Logging saved artifacts to MLflow run ID: {best_run_id}")
        mlflow.log_artifact(BEST_MODEL_FILENAME)
    
        print("Saved artifacts logged as MLflow artifacts.")
    except Exception as e:
        print(f"Error logging artifacts to MLflow: {e}")


mlflow.log_param("best_model_name", best_model_name)
mlflow.log_param("best_model_accuracy", best_accuracy)
mlflow.log_param("best_model_run_id", best_run_id) # Log the run ID of the best model

print("\nMLflow tracking and artifact saving complete.")
print(f"View results in MLflow UI by running 'mlflow ui' in your terminal.")




MLflow tracking URI set to: https://dagshub.com/api/v1/repo-buckets/s3/sandhya-bdb


Accessing as sandhya-bdb

Initialized MLflow to track repo "sandhya-bdb/mlflow_dagshub_new"

Repository sandhya-bdb/mlflow_dagshub_new initialized!

DagsHub and MLflow initialized successfully.
Dataset 'survey_results_op.csv' loaded successfully. Shape: (29956, 20)
First 5 rows of the dataset:
  respondent_id gender  zone            occupation  income_levels  \
0        R00001      M     3  Working Professional              1   
1        R00002      F     4  Working Professional              5   
2        R00003      F     1  Working Professional              5   
3        R00004      F     3  Working Professional              3   
4        R00005      M     4               Student              0   

   consume_frequency(weekly) current_brand preferable_consumption_size  \
0                          2      Newcomer             Medium (500 ml)   
1                          3   Established             Medium (500 ml)   
2                          2      Newcomer             Medium (500 ml)   
3                          3      Newcomer             Medium (500 ml)   
4                          2   Established             Medium (500 ml

/opt/anaconda3/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [20:26:27] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  Accuracy: 0.9246
  Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.91      0.91      1930
           1       0.90      0.91      0.91      2223
           2       0.96      0.95      0.95      2430
           3       0.92      0.92      0.92       906

    accuracy                           0.92      7489
   macro avg       0.92      0.92      0.92      7489
weighted avg       0.92      0.92      0.92      7489

------------------------------
🏃 View run Train_XGBoost at: https://dagshub.com/sandhya-bdb/mlflow_dagshub_new.mlflow/#/experiments/0/runs/4180e26398c9423c96e6c8067027777b
🧪 View experiment at: https://dagshub.com/sandhya-bdb/mlflow_dagshub_new.mlflow/#/experiments/0

Training Light Gradient Boosting Machine (LightGBM)...
MLflow Run ID: 627cdebed2ab462c99ff667ba53061c2
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhea